# Dicionário do Português: first look

Runs on the full data on Kaggle or on the CSV sample of the GitHub repository (the 1,000 most frequent words).
The AI texts were **not reviewed**: read them as what `text-davinci-002` and `gpt-3.5-turbo` wrote, not as a reference.

In [1]:
from pathlib import Path
import duckdb

K = Path("/kaggle/input/dicionario-portugues")
if K.exists():
    P, T, R, F = (f"{K}/{n}-*.parquet" for n in ("palavras", "textos_ia", "rimas", "mesmo_fingerprint"))
else:
    S = Path("../sample")
    P, T, R, F = (str(S / f"{n}_sample.csv") for n in ("palavras", "textos_ia", "rimas", "mesmo_fingerprint"))
duckdb.sql(f"SELECT count(*) AS palavras, count(*) FILTER (tem_textos_ia) AS com_textos_ia FROM '{P}'")

┌──────────┬───────────────┐
│ palavras │ com_textos_ia │
│  int64   │     int64     │
├──────────┼───────────────┤
│     1000 │          1000 │
└──────────┴───────────────┘

## Most frequent words

In [2]:
duckdb.sql(f"SELECT palavra, ocorrencias, numero_letras, metaphone, classe_gramatical FROM '{P}' ORDER BY ocorrencias DESC LIMIT 15")

┌──────────┬─────────────┬───────────────┬───────────┬─────────────────────────────────────────────────────┐
│ palavra  │ ocorrencias │ numero_letras │ metaphone │                  classe_gramatical                  │
│ varchar  │    int64    │     int64     │  varchar  │                       varchar                       │
├──────────┼─────────────┼───────────────┼───────────┼─────────────────────────────────────────────────────┤
│ Fazer    │      576699 │             5 │ FZ2       │ Verbo                                               │
│ Curso    │      402004 │             5 │ KRS       │ Nome próprio                                        │
│ Online   │      381577 │             6 │ ONLN      │ A palavra "online" é um adjetivo.                   │
│ Preço    │      377652 │             4 │ PRS       │ Preço é um substantivo.                             │
│ Brasil   │      350644 │             6 │ BRZL      │ Brasil é um substantivo masculino.                  │
│ Vida     │      3

## Words that sound alike (same Metaphone key)

In [3]:
duckdb.sql(f"""
    SELECT metaphone, count(*) AS palavras, string_agg(palavra, ', ' ORDER BY ocorrencias DESC) AS exemplos
    FROM '{P}' GROUP BY metaphone HAVING count(*) > 1 ORDER BY palavras DESC LIMIT 10""")

┌───────────┬──────────┬─────────────────────────────────────────────────────┐
│ metaphone │ palavras │                      exemplos                       │
│  varchar  │  int64   │                       varchar                       │
├───────────┼──────────┼─────────────────────────────────────────────────────┤
│ PS        │        8 │ Passo, Pessoa, Peças, Pois, Posso, Paz, Pais, Passa │
│ PRT       │        7 │ Porto, Preto, Porta, Preta, Parte, Prata, Perto     │
│ P         │        6 │ Pai, Pe, Pão, Pó, Pao, Pé                           │
│ M         │        5 │ Meio, Mae, Mãe, Meia, Mão                           │
│ BL        │        5 │ Bolo, Belo, Bula, Bola, Bela                        │
│ FS        │        5 │ Faz, Faça, Fez, Foz, Face                           │
│ SL        │        5 │ Sul, Sol, Sala, Salão, Solo                         │
│ KRT       │        5 │ Cartão, Corte, Quarto, Carta, Curto                 │
│ VL        │        4 │ Vila, Vale, Vou, Violão    

## Rhymes

In [4]:
duckdb.sql(f"""
    SELECT r.palavra, string_agg(r.rima, ', ') AS rimas FROM '{R}' r LEFT JOIN '{P}' p ON p.palavra = r.palavra
    GROUP BY r.palavra, p.ocorrencias ORDER BY p.ocorrencias DESC NULLS LAST, r.palavra LIMIT 10""")

┌──────────────┬────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┐
│   palavra    │                                                             rimas                                                              │
│   varchar    │                                                            varchar                                                             │
├──────────────┼────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┤
│ Tonificáveis │ Abaláveis, Abandonáveis, Abandalháveis, Abaixáveis, Abalizáveis, Abafáveis, Abalroáveis, Abarcáveis, Abanáveis, Abalançáveis   │
│ Tonifique    │ Classifique, Certifique, Amplifique, Codifique, Clarifique, Corporifique, Beatifique, Calcifique, Bestifique, Acidifique       │
│ Tonifiquei   │ Apliquei, Abdiquei, Autentiquei, Adociquei, Beatifiquei, Adjudiquei, Acidifiquei, Barriquei, Amplifiquei, B

## Same-key pairs: inflections and some unrelated words

In [5]:
duckdb.sql(f"""
    SELECT palavra, count(*) AS pares, array_to_string(list(outra)[1:6], ', ') AS exemplos
    FROM '{F}' GROUP BY palavra ORDER BY pares DESC LIMIT 8""")

┌────────────┬───────┬─────────────────────────────────────────────────────────────────────────────────┐
│  palavra   │ pares │                                    exemplos                                     │
│  varchar   │ int64 │                                     varchar                                     │
├────────────┼───────┼─────────────────────────────────────────────────────────────────────────────────┤
│ Falas      │   147 │ Feliz, Falar, Fala, Falso, Falando, Falsa                                       │
│ Mudaram    │    92 │ Mudar, Muda, Mudas, Mudo, Mudou, Mude                                           │
│ Roles      │    90 │ Relação, Rolo, Relações, Role, Rolar, Rolos                                     │
│ Ameaças    │    85 │ Ameaça, Amas, Ameaçado, Amasse, Amos, Ame-se                                    │
│ Atraem     │    81 │ Atrair, Atrai, Atraído, Atraido, Atraindo, Atraídos                             │
│ Apresentam │    79 │ Apresentação, Apresenta, Apresen

## AI texts by field and model

In [6]:
duckdb.sql(f"""
    SELECT campo, count(*) FILTER (modelo = 'gpt-3.5-turbo') AS gpt35, count(*) FILTER (modelo = 'text-davinci-002') AS davinci
    FROM '{T}' GROUP BY campo ORDER BY gpt35 + davinci DESC""")

┌─────────────────────────────────┬───────┬─────────┐
│              campo              │ gpt35 │ davinci │
│             varchar             │ int64 │  int64  │
├─────────────────────────────────┼───────┼─────────┤
│ silaba_tonica                   │     4 │     196 │
│ feminino                        │     4 │     196 │
│ sinonimos                       │     4 │     196 │
│ gerundio                        │     4 │     196 │
│ separacao_silabica              │     4 │     196 │
│ significado_em_contexto         │     4 │     196 │
│ participio                      │     4 │     196 │
│ classe_gramatical               │     4 │     196 │
│ definicoes                      │     4 │     196 │
│ plural                          │     4 │     196 │
│ singular                        │     4 │     196 │
│ masculino                       │     4 │     196 │
│ significado                     │     4 │     196 │
│ exemplos_de_uso                 │     4 │     196 │
│ antonimos                 

## How part of speech was answered

The field is free text: one word (*Verbo*), a sentence, or English (*Noun*). A first normalization:

In [7]:
duckdb.sql(f"""
    SELECT CASE
             WHEN regexp_matches(lower(classe_gramatical), 'noun|verb\\b|adjective') THEN 'English answer'
             WHEN regexp_matches(lower(classe_gramatical), 'nome próprio|substantivo próprio') THEN 'nome próprio'
             WHEN regexp_matches(lower(classe_gramatical), 'verbo') THEN 'verbo'
             WHEN regexp_matches(lower(classe_gramatical), 'substantivo|^nome') THEN 'substantivo'
             WHEN regexp_matches(lower(classe_gramatical), 'adjetivo') THEN 'adjetivo'
             WHEN regexp_matches(lower(classe_gramatical), 'advérbio') THEN 'advérbio'
             WHEN regexp_matches(lower(classe_gramatical), 'pronome') THEN 'pronome'
             ELSE 'other' END AS classe, count(*) AS palavras
    FROM '{P}' WHERE classe_gramatical IS NOT NULL GROUP BY classe ORDER BY palavras DESC""")

┌────────────────┬──────────┐
│     classe     │ palavras │
│    varchar     │  int64   │
├────────────────┼──────────┤
│ substantivo    │      313 │
│ nome próprio   │      274 │
│ English answer │      215 │
│ verbo          │       89 │
│ adjetivo       │       42 │
│ other          │       33 │
│ pronome        │       23 │
│ advérbio       │       10 │
└────────────────┴──────────┘

## A known failure: definitions that repeat the word

Count meanings that start with *"X significa X"*, a sign of an empty answer.

In [8]:
duckdb.sql(f"""
    SELECT palavra, left(significado, 90) AS significado FROM '{P}'
    WHERE regexp_matches(lower(significado), '^' || lower(palavra) || ' significa ' || lower(palavra) || '\\b')
    ORDER BY ocorrencias DESC LIMIT 10""")

┌───────────┬────────────────────────────────────────────────────────────────────────────────────────────┐
│  palavra  │                                        significado                                         │
│  varchar  │                                          varchar                                           │
├───────────┼────────────────────────────────────────────────────────────────────────────────────────────┤
│ Fazer     │ Fazer significa fazer algo, geralmente com as mãos. Pode significar construir, reparar ou  │
│ Criar     │ Criar significa criar ou produzir algo novo. Pode ser usado para descrever a atividade de  │
│ Todo      │ Todo significa todo ou qualquer um. Pode ser usado como um adjetivo ou pronome para indica │
│ Encontrar │ Encontrar significa encontrar, achar ou descobrir algo ou alguém. Pode ser usado para desc │
│ Aumentar  │ Aumentar significa aumentar ou aumentar a quantidade de algo.                              │
└───────────┴────────────────────────